# Intro to RAG pipelines

In [5]:
!uv add langchain-ollama

Resolved 75 packages in 1ms
Checked 68 packages in 0.59ms


In [8]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="ornith-1.5:9b", base_url="http://localhost:11434")

In [9]:
result = llm.invoke("Quem é Sherlock Holmes?")
print(result)

content='**Sherlock Holmes** é um detetive fictício criado pelo escritor escocês **Sir Arthur Conan Doyle**, e primeiro apareceu em 1887 na obra *"Um Estudo em Escarlate"* (A Study in Scarlet).\n\n## Características principais\n\n**Perfil**\n- Detetive consultor que vive em **221B Baker Street**, em Londres\n- Conhecido por sua **lucidez mental extraordinária**, capacidade de **observação** e **dedução lógica**\n- Famoso por resolver casos criminais e enigmas aparentemente impossíveis\n- Costuma declarar que "a diferença entre um homem bem educado e um bem informado é que o segundo pode explicar o que está sabendo"\n\n**Personagens associados**\n- **Dr. John Watson** — seu companheiro, médico e cronista das aventuras\n- **Professor Moriarty** — o célebre vilão que Holmes descreveu como "o Napoleão do crime"\n- **Dr. Joseph Bell** — médico real que inspirou o personagem\n\n## Obras mais famosas\n\nAlguns dos casos mais conhecidos incluem:\n- *Um Estudo em Escarlate*\n- *O Caso do Coraçã

In [11]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    base_url="http://localhost:11434", model="nomic-embed-text-v2-moe:latest"
)

In [12]:
result = embedding_model.embed_query("Quem é Sherlock Holmes?")
print(result)

[0.030467078, 0.018971743, -0.04616939, -0.00039493555, 0.029336594, -0.008759227, -0.050811287, 0.029400405, -0.015603257, 0.020610759, -0.029169, -0.03652877, 0.053583793, 0.0008754857, 0.023672009, -0.08062577, 0.029106839, -0.0020052097, 0.025747642, 0.05039944, 0.057666503, 0.032947943, 0.03994148, -0.024195524, 0.0025727483, 0.02378963, 0.026163025, -0.012814708, 0.04709246, 0.0328907, -0.010671738, 0.03690868, -0.0005314478, 0.0119155655, -0.0020633428, -0.0036863287, -0.039323654, 0.02599826, 0.00972656, 0.015217034, 0.010822214, 0.0059062387, 0.015161357, 0.013942309, 0.017237354, 0.0035914085, -0.06552999, 0.031190615, 0.019838214, -0.012030245, 0.019924337, -0.0017590455, -0.031371847, -0.04604895, 0.038150284, -0.057288263, 0.055358555, -0.028026134, 0.029123403, 0.031647716, -0.016594106, 0.033291593, 0.0941157, -0.024832457, -0.011815017, -0.032475565, -0.006024251, 0.01803848, -0.07432407, -0.02371397, -0.018658422, -0.01630455, -0.027646389, 0.045323994, -0.01733225, 0.

# 1. Carregar dados e documentos

In [30]:
import json

with open("booklist.json", "r") as f:
    booklist = json.load(f)

In [53]:
!uv add langchain-community langchain-text-splitters

Resolved 92 packages in 1ms
Checked 85 packages in 0.37ms


In [42]:
from langchain_community.document_loaders import TextLoader
import os

documents = []
for book in booklist:
    loader = TextLoader(book["path"], encoding="utf-8")
    doc = loader.load()[0]
    doc.metadata["title"] = book["title"]
    doc.metadata["author"] = book["author"]
    doc.metadata["year"] = book["year"]
    doc.metadata["genre"] = book["genre"]
    doc.metadata["language"] = book["language"]
    documents.extend([doc])

print(f"Loaded {len(documents)} documents from {len(booklist)} books.")

Loaded 2 documents from 2 books.


In [44]:
print(documents[0].metadata)  # Print the metadata of the first document

{'source': 'books/alices_adventure_in_wonderland.txt', 'title': "Alice's Adventures in Wonderland", 'author': 'Lewis Carroll', 'year': 1865, 'genre': 'Fantasy', 'language': 'English'}


In [55]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=250)
chunks = splitter.split_documents(documents)

In [57]:
print("Total number of chunks created:", len(chunks))

Total number of chunks created: 675


In [60]:
print(chunks[500])  # Print the metadata of the first chunk

page_content='“Forgive me, Doctor; I forgot myself. You do not need any help. I am so
worried in my mind that I am apt to be irritable. If you only knew the
problem I have to face, and that I am working out, you would pity, and
tolerate, and pardon me. Pray do not put me in a strait-waistcoat. I
want to think and I cannot think freely when my body is confined. I am
sure you will understand!” He had evidently self-control; so when the
attendants came I told them not to mind, and they withdrew. Renfield
watched them go; when the door was closed he said, with considerable
dignity and sweetness:--

“Dr. Seward, you have been very considerate towards me. Believe me that
I am very, very grateful to you!” I thought it well to leave him in this
mood, and so I came away. There is certainly something to ponder over in
this man’s state. Several points seem to make what the American
interviewer calls “a story,” if one could only get them in proper order.
Here they are:--

Will not mention “drinkin